This notebook only contains the machine learning model study. The complete EDA can be found at another notebook [https://www.kaggle.com/code/qiaoningchen/stroke-prediction-playground-eda](http://)

This notebook use 3 datasets. Two datasets are from ''Playground Series S3E2''(train.csv, test.csv). The final train data set combines ''train.csv'' and the orginal stroke prediction data "healthcare-dataset-stroke-data.csv". The test data set is "test.csv". For machine learning beginners, the prediction file should contain the stroke probability data not the stroke boolean data. Thanks to JACOPO REPOSSI, I can be able to correct this mistake and my score has tremendously improved.

In [1]:
import pandas as pd 
from warnings import simplefilter

simplefilter('ignore')


stroke_train= pd.read_csv("/kaggle/input/playground-series-s3e2/train.csv")
stroke_test_data = pd.read_csv("/kaggle/input/playground-series-s3e2/test.csv")
original_data = pd.read_csv("/kaggle/input/stroke-prediction-dataset/healthcare-dataset-stroke-data.csv")

In [2]:
original_data.shape

(5110, 12)

In [3]:
original_data.dtypes

id                     int64
gender                object
age                  float64
hypertension           int64
heart_disease          int64
ever_married          object
work_type             object
Residence_type        object
avg_glucose_level    float64
bmi                  float64
smoking_status        object
stroke                 int64
dtype: object

In [4]:
original_data.isnull().sum()

id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

In [5]:
stroke_train_data = pd.concat([stroke_train, original_data]).reset_index(drop=True)

In [6]:
stroke_train.shape

(15304, 12)

In [7]:
stroke_train_data.shape

(20414, 12)

In [8]:
# using dictionary to convert specific columns
convert_dict = {'hypertension': object,
                'heart_disease': object,
                }
 
stroke_train_data = stroke_train_data.astype(convert_dict)
stroke_train_data.dtypes

id                     int64
gender                object
age                  float64
hypertension          object
heart_disease         object
ever_married          object
work_type             object
Residence_type        object
avg_glucose_level    float64
bmi                  float64
smoking_status        object
stroke                 int64
dtype: object

In [9]:
# using dictionary to convert specific columns
convert_dict = {'hypertension': object,
                'heart_disease': object,
                }
 
stroke_test_data = stroke_test_data.astype(convert_dict)
stroke_test_data.dtypes

id                     int64
gender                object
age                  float64
hypertension          object
heart_disease         object
ever_married          object
work_type             object
Residence_type        object
avg_glucose_level    float64
bmi                  float64
smoking_status        object
dtype: object

In [10]:
stroke_test_data.isnull().sum()

id                   0
gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
dtype: int64

In [11]:
# target
y = stroke_train_data.stroke

#features
features = ['gender', 'age','hypertension','heart_disease','ever_married',
            "work_type","Residence_type","avg_glucose_level","bmi","smoking_status"]


# Select columns corresponding to features
X_train_full = stroke_train_data[features]

In [12]:
X_train = pd.get_dummies(X_train_full[features])
X_test = pd.get_dummies(stroke_test_data[features])

In [13]:
# The missing numerical values are from the original data set 
# which is included in X_train. Impute it.

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer


#imputation
my_imputer = SimpleImputer(strategy = 'median') # Your code here
imputed_X_train = pd.DataFrame(my_imputer.fit_transform(X_train))
imputed_X_test = pd.DataFrame(my_imputer.transform(X_test))

# Fill in the lines below: imputation removed column names; put them back
imputed_X_train.columns = X_train.columns
imputed_X_test.columns = X_test.columns

In [14]:
imputed_X_train.shape

(20414, 23)

In [15]:
imputed_X_test.shape

(10204, 23)

In [16]:
imputed_X_train.isnull().sum()

age                               0
avg_glucose_level                 0
bmi                               0
gender_Female                     0
gender_Male                       0
gender_Other                      0
hypertension_0                    0
hypertension_1                    0
heart_disease_0                   0
heart_disease_1                   0
ever_married_No                   0
ever_married_Yes                  0
work_type_Govt_job                0
work_type_Never_worked            0
work_type_Private                 0
work_type_Self-employed           0
work_type_children                0
Residence_type_Rural              0
Residence_type_Urban              0
smoking_status_Unknown            0
smoking_status_formerly smoked    0
smoking_status_never smoked       0
smoking_status_smokes             0
dtype: int64

In [17]:
imputed_X_test.isnull().sum()

age                               0
avg_glucose_level                 0
bmi                               0
gender_Female                     0
gender_Male                       0
gender_Other                      0
hypertension_0                    0
hypertension_1                    0
heart_disease_0                   0
heart_disease_1                   0
ever_married_No                   0
ever_married_Yes                  0
work_type_Govt_job                0
work_type_Never_worked            0
work_type_Private                 0
work_type_Self-employed           0
work_type_children                0
Residence_type_Rural              0
Residence_type_Urban              0
smoking_status_Unknown            0
smoking_status_formerly smoked    0
smoking_status_never smoked       0
smoking_status_smokes             0
dtype: int64

In [18]:
from xgboost import XGBClassifier

#model
my_model = XGBClassifier()
my_model.fit(X_train,y)

# predictions
# Here we use predict_proba(X_test)[:,1] instead of predict!!!!
# A great reference can be found at
#https://www.kaggle.com/code/jacoporepossi/tutorial-roc-auc-clearly-explained#Model-thresholds


my_list = my_model.predict_proba(X_test)[:,1]

In [19]:
my_list

array([3.1514600e-02, 7.3234566e-02, 1.4133721e-04, ..., 4.4707926e-05,
       6.1532843e-04, 1.0288763e-04], dtype=float32)

In [20]:
#Use cross validation to evaluate the performance of the model
from sklearn.model_selection import cross_val_score
scores = cross_val_score(my_model, X_train, y, cv=5)
print("Mean cross-validation score: %.2f" % scores.mean())

Mean cross-validation score: 0.95


In [21]:
# output predictions data
output = pd.DataFrame({'id': stroke_test_data.id, 'stroke': my_list})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!
